# A3: Predicting Car Price as Four Classes
Student ID: 126671. Built from the course multinomial logistic regression and the A2 car-price preprocessing. Run from the Assignment 3 directory.

## 1. Preparation
As in A2: remove Test Drive Cars and CNG/LPG, extract the brand from the name, strip measurement units, and retain maximum power, mileage, year and brand. To prevent leakage, category boundaries, imputation, brand encoding and scaling are estimated from the training rows. Unknown test/app brands map to the most frequent training brand. Price categories are quartiles 0–3 of training selling prices; exact price cutoffs are printed below.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from model import LogisticRegression, prepare_data

X_train, X_test, y_train, y_test, prep = prepare_data("Cars.csv")
print("Train / test:", X_train.shape, X_test.shape)
print("Category price cutoffs:", prep["edges"])
print("Train class support:", np.bincount(y_train, minlength=4))
print("Test class support:", np.bincount(y_test, minlength=4))

## 2. Extended course logistic regression
`model.py` adapts the course `LogisticRegression` softmax class. Softmax subtracts the maximum logit for numerical stability. The loss and gradient use the *mean* cross entropy. Ridge adds λ·ΣW² over the feature weights (not the bias) and adds 2λW to their gradient; set `ridge=False` to disable it. Per-class precision, recall, F1 and support are computed from TP/FP/FN, then macro and support-weighted averages are calculated from these values. Support means the number of actual examples of each class.

In [ ]:
model = LogisticRegression(method="batch", alpha=0.3, max_iter=1200, ridge=True, lam=0.001).fit(X_train, y_train)
predictions = model.predict(X_test)
print("Our report:", model.report(y_test, predictions))
print("sklearn report:\n", classification_report(y_test, predictions, labels=[0,1,2,3], zero_division=0))

## 3. Independent metric check
Compare all computed scores with scikit-learn at full floating-point precision, including support and macro/weighted scores. The assignment PDF contains two typographical issues: weighted scores are **not** divided by four after applying proportions, and F1 is 2PR/(P+R).

In [ ]:
reference = classification_report(y_test, predictions, labels=[0,1,2,3], output_dict=True, zero_division=0)
ours = model.report(y_test, predictions)
for c in range(4):
    for ours_key, sklearn_key in [("precision","precision"),("recall","recall"),("f1","f1-score"),("support","support")]:
        assert np.isclose(ours["per_class"][c][ours_key], reference[str(c)][sklearn_key])
for average, ref_name in [("macro","macro avg"),("weighted","weighted avg")]:
    for metric, ref_metric in [("precision","precision"),("recall","recall"),("f1","f1-score")]:
        assert np.isclose(ours[f"{average}_{metric}"], reference[ref_name][ref_metric])
assert np.isclose(ours["accuracy"], reference["accuracy"])
print("All manual metrics match scikit-learn.")

## 4. Experiments and model saving
Four runs vary the ridge flag and penalty strength. Select the penalty by macro F1 on a stratified validation split within the training data; then refit the best configuration on all training rows and report the held-out test score once. `train.py` saves the selected model and preprocessing bundle in `app/model.pkl` and writes `experiment_results.json`. Run the final cell with the remote URI to log metrics and a model artifact. Never log the dataset.

In [ ]:
from train import run
bundle = run("Cars.csv")

### Remote tracking and registry steps
Run `run("Cars.csv", tracking_uri="http://mlflow.ml.brain.cs.ait.ac.th/")` only when connected to the AIT network and after installing `mlflow`. In the MLflow UI, open experiment `126671-a3`, inspect the runs, select the best model artifact, register it as `126671-a3-model`, and set its version to Staging as requested in the brief. Record its run ID, model version and deployment URL in the README. The course server may use legacy stage controls; follow its UI.

## 5. App, tests, CI/CD
`app/app.py` displays one of four price ranges. `tests/test_model.py` checks the expected input and output shape. The root `.github/workflows/a3.yml` runs tests after a push and contains a gated SSH deployment job. Server secrets and the AIT network/ProxyJump path must be configured before remote deployment can occur. See README for commands.